In [ ]:
import sys

print(sys.executable)

In [ ]:
import trimesh
import plotly.graph_objects as go

print("trimesh:", trimesh.__version__)
print("Plotly loaded successfully")

In [ ]:
import trimesh
from pathlib import Path

# Explicit ResQNet project path
PROJECT_ROOT = Path("/home/rishika/ResQNet")

obj_path = PROJECT_ROOT / "sionna" / "scenes" / "city_damaged.obj"

print("Loading:", obj_path)
print("File exists:", obj_path.exists())

mesh = trimesh.load_mesh(str(obj_path))

print()
print("3D MODEL LOADED")
print("Vertices:", len(mesh.vertices))
print("Faces:", len(mesh.faces))
print("Bounds:")
print(mesh.bounds)

In [ ]:
import plotly.graph_objects as go
import numpy as np

vertices = mesh.vertices
faces = mesh.faces

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            opacity=0.75,
            flatshading=True,
            name="Damaged City"
        )
    ]
)

fig.update_layout(
    title="ResQNet — 3D Damaged City",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Height (m)",
        aspectmode="data",
    ),
    height=750,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

In [ ]:
# PPO-selected UAV position
uav_x = -18.82
uav_y = 48.41
uav_z = 30.57

# Create the UAV marker
uav_trace = go.Scatter3d(
    x=[uav_x],
    y=[uav_y],
    z=[uav_z],
    mode="markers+text",
    marker=dict(
        size=10,
        symbol="diamond",
    ),
    text=["🚁 PPO UAV"],
    textposition="top center",
    name="PPO UAV"
)

# Add UAV to the existing city figure
fig.add_trace(uav_trace)

fig.update_layout(
    title="ResQNet — Damaged City + PPO UAV"
)

fig.show()

In [ ]:
# Survivor / user positions used in the network simulation
users = [
    [-20, -30, 1.5],
    [-10, -20, 1.5],
    [0, -10, 1.5],
    [10, 0, 1.5],
    [20, 10, 1.5]
]

user_x = [p[0] for p in users]
user_y = [p[1] for p in users]
user_z = [p[2] for p in users]

survivor_trace = go.Scatter3d(
    x=user_x,
    y=user_y,
    z=user_z,
    mode="markers+text",
    marker=dict(
        size=7,
        symbol="circle",
    ),
    text=[
        "Survivor 1",
        "Survivor 2",
        "Survivor 3",
        "Survivor 4",
        "Survivor 5"
    ],
    textposition="top center",
    name="Survivors"
)

fig.add_trace(survivor_trace)

fig.update_layout(
    title="ResQNet — Damaged City + PPO UAV + Survivors"
)

fig.show()

In [ ]:
import sys
from pathlib import Path

# Make the Sionna pipeline available
PROJECT_ROOT = Path("/home/rishika/ResQNet")
sys.path.insert(0, str(PROJECT_ROOT / "sionna"))

import sionna_pipeline as p

# PPO-selected UAV
ppo_uav = [[-18.82, 48.41, 30.57]]

# Survivor locations
survivor_positions = [
    [-20, -30, 1.5],
    [-10, -20, 1.5],
    [0, -10, 1.5],
    [10, 0, 1.5],
    [20, 10, 1.5]
]

# Run actual Sionna RT propagation on the damaged city
sionna_data, coverage_grid = p.run_uav_propagation_sim(
    ppo_uav,
    survivor_positions,
    scene_name=str(
        PROJECT_ROOT / "sionna" / "scenes" / "city_damaged.xml"
    )
)

print("SIONNA RT PROPAGATION COMPLETE")
print()
print("PPO UAV:", ppo_uav[0])
print(
    "Sionna coverage:",
    sionna_data["ml_summary_metrics"]["coverage_percentage"],
    "%"
)
print(
    "Mean path gain:",
    sionna_data["ml_summary_metrics"]["mean_path_gain_db"],
    "dB"
)

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path("/home/rishika/ResQNet")

# PPO UAV position
ppo_uav = [-18.82, 48.41, 30.57]

# Sionna results for the PPO UAV
sionna_result = {
    "coverage_percentage": 0.56,
    "mean_path_gain_db": -79.2
}

print("3D DEMO DATA READY")
print("PPO UAV:", ppo_uav)
print("Sionna coverage:", sionna_result["coverage_percentage"], "%")
print("Mean path gain:", sionna_result["mean_path_gain_db"], "dB")

In [ ]:
import plotly.graph_objects as go
import trimesh
import numpy as np

# Load damaged city model
obj_path = "/home/rishika/ResQNet/sionna/scenes/city_damaged.obj"
mesh = trimesh.load(obj_path, force="mesh")

vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.faces)

# PPO-selected UAV
uav = np.array([-18.82, 48.41, 30.57])

# Create 3D city mesh
fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    opacity=0.55,
    name="Damaged City"
))

# Add UAV
fig.add_trace(go.Scatter3d(
    x=[uav[0]],
    y=[uav[1]],
    z=[uav[2]],
    mode="markers+text",
    marker=dict(
        size=10,
        color="red"
    ),
    text=["PPO UAV"],
    textposition="top center",
    name="PPO UAV"
))

# UAV vertical connection to ground
fig.add_trace(go.Scatter3d(
    x=[uav[0], uav[0]],
    y=[uav[1], uav[1]],
    z=[0, uav[2]],
    mode="lines",
    line=dict(width=4),
    name="UAV altitude"
))

fig.update_layout(
    title="ResQNet — 3D Damaged City + PPO UAV Placement",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Altitude",
        aspectmode="data"
    ),
    height=750
)

fig.show()